# 02 — Feature Engineering & Frequency Alignment

Build the **daily feature matrix** and train/val/test splits for all models, organised
into explicit **ablation groups**. Monthly macro is intentionally *not* here — its
leak-corrected weekly lags live in `02d_macro_features_weekly.ipynb` (publication-lag
aware). This notebook stays daily and observable-only.

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120

## 1. Load raw data

In [ ]:
prices = pd.read_csv('../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
fred_d = pd.read_csv('../data/raw/daily_fred.csv',   index_col=0, parse_dates=True)

prices.index = prices.index.tz_localize(None)   # strip tz if present
fred_d.index = fred_d.index.tz_localize(None)

# Align FRED daily onto the yfinance trading-day grid. ffill first (fred_d's index is the
# union of 3 calendars — DFII10/T10YIE on business days + ICSA on Saturdays — so each row
# has NaN for the series whose calendar doesn't include it), THEN reindex with
# method='ffill' for any missing target dates. (Full rationale: 01_eda §1.)
prices = prices.join(fred_d.ffill().reindex(prices.index, method='ffill'))
print('Daily grid:', prices.shape)

## 2. Daily features

All transforms are model-ready and observable same-day (weekly notebooks re-aggregate to
W-FRI and add their own 1-week lag). Change conventions match `01_eda` §6/§7: prices →
log-return, rates → Δ (yield change), counts → Δlog.

In [ ]:
df = pd.DataFrame(index=prices.index)

# ── Target ───────────────────────────────────────────────────────────────────
df['silver']        = prices['silver']                  # level — kept for tech features (02c)
df['silver_return'] = np.log(prices['silver']).diff()   # log-return (weekly notebooks W-FRI-sum)

# ── YF_DAILY: cross-asset log-returns ─────────────────────────────────────────
for a in ['gold', 'copper', 'usd_index', 'sp500', 'vix', 'oil']:
    df['usd_return' if a == 'usd_index' else f'{a}_return'] = np.log(prices[a]).diff()

# ── GS: gold/silver ratio z-score (train-period stats only — no look-ahead) ───
# The ratio hovers ~80 across the sample, so a fixed pre-2022 baseline captures
# "stretched vs the long-run norm" better than a rolling window (which drifts during
# long deviations like Mar 2020). Train = pre-2022; same mean/std applied to val + test.
_gs       = prices['gold'] / prices['silver']
_gs_train = _gs[_gs.index < '2022-01-01']
df['gs_ratio_z'] = (_gs - _gs_train.mean()) / _gs_train.std()
print(f'gs_ratio_z: train mean={_gs_train.mean():.2f}, std={_gs_train.std():.2f}')

# ── FRED_DAILY: rates as Δyield, jobless claims as Δlog ───────────────────────
df['real_rates_chg'] = prices['real_rates_10y'].diff()
df['breakeven_chg']  = prices['breakeven_10y'].diff()
df['jobless_chg']    = np.log(prices['jobless_claims']).diff()

# ── AR (own-history / weak-form): silver-return lags ──────────────────────────
for lag in [1, 2, 3]:
    df[f'silver_lag{lag}'] = df['silver_return'].shift(lag)

print(df.shape)
df.head()

## 3. COT positioning — COMEX silver disaggregated report

Weekly Commitments-of-Traders data from the CFTC (`disaggregated_fut`, collected via
`src/collect_cot.py` → `data/raw/cot_silver.csv`). Two net-positioning series:

- `cot_mm_net_pct`   = (Managed Money longs − shorts) / Open Interest — speculative positioning
- `cot_comm_net_pct` = (Producer/Merchant longs − shorts) / Open Interest — hedger positioning

**Lag rigour**: CFTC publishes Friday afternoon with data as-of the preceding Tuesday. The
report dated Friday $t-1$ carries Tuesday $t-1$ data — fully observed before a Friday-$t$
target, so a 1-week lag is leak-safe. Here we forward-fill the release snapshot onto the
daily grid; weekly notebooks add the lag.

In [ ]:
cot_path = '../data/raw/cot_silver.csv'
if os.path.exists(cot_path):
    cot = pd.read_csv(cot_path, parse_dates=['report_date']).sort_values('report_date').set_index('report_date')
    mm_net   = cot['M_Money_Positions_Long_All']   - cot['M_Money_Positions_Short_All']
    comm_net = cot['Prod_Merc_Positions_Long_All'] - cot['Prod_Merc_Positions_Short_All']
    oi       = cot['Open_Interest_All']
    cot_feats = pd.DataFrame({'cot_mm_net_pct': mm_net / oi, 'cot_comm_net_pct': comm_net / oi})
    df = df.join(cot_feats.reindex(df.index, method='ffill'))
    print(f'COT merged ({cot.index.min().date()} → {cot.index.max().date()}).')
else:
    print('cot_silver.csv missing — run `python src/collect_cot.py`.')
    df['cot_mm_net_pct'] = np.nan
    df['cot_comm_net_pct'] = np.nan

## 4. Sentiment (Reddit + news)

Daily FinBERT (news) + Twitter-RoBERTa (Reddit) scores from `03_sentiment.ipynb`:
`reddit_sentiment`, `news_sentiment`, and the combined `sentiment_score`. The auxiliary
count/weight columns are consumed directly from `daily_sentiment.csv` by the volatility
notebook, so they don't enter the split. `SENT` ablates the reddit+news pair;
`sentiment_score` is the combined daily index (kept for the daily LSTM).

In [ ]:
sentiment_path = '../data/processed/daily_sentiment.csv'
SENT_COLS = ['reddit_sentiment', 'news_sentiment', 'sentiment_score']
if os.path.exists(sentiment_path):
    sent = pd.read_csv(sentiment_path, index_col=0, parse_dates=True)
    df = df.join(sent[SENT_COLS], how='left')
    print('Sentiment joined:', SENT_COLS)
else:
    print('No sentiment file — run 03_sentiment.ipynb first.')
    for c in SENT_COLS:
        df[c] = np.nan

## 5. Feature groups for ablations

Explicit column groups so model notebooks select features by group name rather than
hardcoding lists. Saved as `data/processed/feature_groups.json`. Monthly macro is absent by
design — see `02d_macro_features_weekly.ipynb` for its leak-corrected weekly lags.

In [ ]:
FEATURE_GROUPS = {
    'AR':         ['silver_lag1', 'silver_lag2', 'silver_lag3'],
    'GS':         ['gs_ratio_z'],
    'YF_DAILY':   ['gold_return', 'copper_return', 'usd_return', 'sp500_return', 'vix_return', 'oil_return'],
    'FRED_DAILY': ['real_rates_chg', 'breakeven_chg', 'jobless_chg'],
    'COT':        ['cot_mm_net_pct', 'cot_comm_net_pct'],
    'SENT':       ['reddit_sentiment', 'news_sentiment'],
}
missing = [c for g in FEATURE_GROUPS.values() for c in g if c not in df.columns]
print('Missing grouped columns:', missing if missing else 'none')
for g, cols in FEATURE_GROUPS.items():
    print(f'  {g:11s} ({len(cols)}): {cols}')

with open('../data/processed/feature_groups.json', 'w') as f:
    json.dump(FEATURE_GROUPS, f, indent=2)
print('\nSaved -> data/processed/feature_groups.json')

## 6. Train / validation / test split
- Train: 2015–2021
- Val:   2022
- Test:  2023–2026 (YTD)

In [ ]:
df = df.dropna(subset=['silver_return'])

train = df[df.index < '2022-01-01']
val   = df[(df.index >= '2022-01-01') & (df.index < '2023-01-01')]
test  = df[df.index >= '2023-01-01']

print(f'Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} days)')
print(f'Val:   {val.index[0].date()} → {val.index[-1].date()}  ({len(val)} days)')
print(f'Test:  {test.index[0].date()} → {test.index[-1].date()}  ({len(test)} days)')
print('Columns:', list(df.columns))

## 7. Save feature matrix

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/features.csv')
train.to_csv('../data/processed/train.csv')
val.to_csv('../data/processed/val.csv')
test.to_csv('../data/processed/test.csv')
print('Saved feature matrix and splits to data/processed/')